## 如何将嵌入向量存储到数据库当中，继而能够进行查询

### 1、什么是向量数据库以及为什么需要使用它
1. 向量数据库就是用来存储向量的数据库。（其他数据库：关系型数据库，列式存储数据库，图数据库，内存数据库。。）
2. 为什么还需要使用向量库：用来做语义搜索。

## 2、有哪些向量数据库可以供我们选择
1. FAISS：是一个可以用来做向量检索的库（library），并不是一个向量数据库服务，并不会监听端口供客户端去查询其内部的数据
2. Chroma：向量数据库（会监听端口，并接收客户端的连接查询），轻量级的向量数据库。在扩展能力上面较不足。
3. **Milvus**: 向量数据库，生产级别（更加稳定）向量数据库。支持横向扩展。**我们需要讲解的重点数据库**

## 3、怎么去使用Milvus（换句话说，通过哪些步骤去构建一个数据库实例）

### 1、在Milvus构建一个Collection（类似于我们在MySQL当中的表）

In [1]:
from pymilvus import MilvusClient
## 获取到客户端对象
client = MilvusClient(
    uri="./mivus_demo.db"
)

/home/m1881/.pyenv/versions/3.12.12/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


### 2、Milvus支持存储除了向量外的其他业务字段。所以接下来需要定义这个Collection的schema信息（schema也就是说，我们的数据结构，有哪些字段）

In [14]:
from pymilvus import DataType
def build_schema():
    schema = MilvusClient.create_schema(auto_id = True)
    schema.add_field(field_name="id",datatype=DataType.INT64,is_primary=True)
    schema.add_field(field_name="vector",datatype=DataType.FLOAT_VECTOR,dim=768)
    schema.add_field(field_name="metadata",datatype=DataType.JSON) # 元数据信息可以用来做检索
    schema.add_field(field_name="content",datatype=DataType.VARCHAR)
    return schema
build_schema()

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>}, {'name': 'metadata', 'description': '', 'type': <DataType.JSON: 23>}, {'name': 'content', 'description': '', 'type': <DataType.VARCHAR: 21>}], 'enable_dynamic_field': False}

### 3、我们需要去定义构建索引的逻辑。索引：索引类似于一本书的目录，能够加速数据查询。

In [15]:
def build_index():
    # 得到索引参数
    index_params = MilvusClient.prepare_index_params()
    # 通过调用索引参数的add_index方法，为collection当中定义的字段去构建索引
    index_params.add_index(
        field_name="vector",
        index_type="AUTOINDEX", # milvus自动根据当前数据量大小选择合适的索引,
        metric_type= "COSINE" # 衡量两个向量之间距离的方式，L2表示的是向量之间的欧式距离
    )
    return index_params
build_index()

[{'field_name': 'vector', 'index_type': 'AUTOINDEX', 'index_name': '', 'metric_type': 'COSINE'}]

### 4、将通过embedding model获取到的向量存储到collection当中

In [ ]:
# 创建 collection
if client.has_collection(collection_name="demo_collection"):
   # 删除 collection
    # 在 Milvus 中删除数据后，存储空间不会立即释放。虽然删除数据会将实体标记为 "逻辑删除"，但实际空间可能不会立即释放。
    # Milvus 会在后台自动压缩数据。这个过程会将较小的数据段合并为较大的数据段，并删除"逻辑删除"的数据或已超过有效时间的数据。
    # 一个名为 Garbage Collection (GC) 的独立进程会定期删除这些 "已删除 "的数据段，从而释放它们占用的存储空间。
    client.drop_collection(collection_name="demo_collection")
client.create_collection(
    collection_name="demo_collection",  # collection 名称
    schema=build_schema(),  # collection 的 schema
    index_params=build_index(),  # collection 的 index
)

# 查看 collection
print(client.list_collections())

# 查看 collection 描述
pprint(client.describe_collection(collection_name="demo_collection"))

### 4、将通过embedding model获取到的向量存储到collection当中
### 5、在做语义检索时，通过milvus内置的检索算法，可以检索到相关的向量及其他内容。此处我们需要学习如何调用milvus api做检索


In [ ]:
## 参见脚本文件：insert_data_to_milvus.py && milvus_query.py
from PIL.ImageText import